# Renz WORM Proxy — Free GPU on Google Colab

Runs the WORM proxy on a free Colab GPU. Exposes it via ngrok. The proxy stays alive as long as this tab is open.

**Setup (one-time):**
1. Sign up for free ngrok account: https://dashboard.ngrok.com/signup
2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it in the cell below

**Run:**
1. Click Runtime → Run all (Ctrl+F9)
2. Wait ~2 minutes for setup
3. Copy the public URL at the bottom (looks like `https://xxxx.ngrok-free.app`)
4. Use that as your endpoint in any OpenAI-compatible client

In [ ]:
# STEP 1: Configuration — paste your ngrok authtoken here
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"  # Get from https://dashboard.ngrok.com/get-started/your-authtoken

# Default persona (NOVA.txt is the most-tested)
PERSONA_FILE = "NOVA.txt"

# Proxy port
PROXY_PORT = 11435

print("Config:")
print(f"  ngrok token: {'SET' if NGROK_AUTHTOKEN != 'PASTE_YOUR_NGROK_TOKEN_HERE' else 'NOT SET — paste it above'}")
print(f"  persona: {PERSONA_FILE}")
print(f"  port: {PROXY_PORT}")

In [ ]:
# STEP 2: Install dependencies
import subprocess
import sys
import os

print("Installing pyngrok...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
print("Done.")

In [ ]:
# STEP 3: Download Renz WORM Proxy
import urllib.request
import os

PROXY_URL = "https://raw.githubusercontent.com/BBRenxo/renz-launcher/main/proxy_server.py"
PERSONA_URL = f"https://raw.githubusercontent.com/BBRenxo/renz-launcher/main/personas/{PERSONA_FILE}"

print(f"Downloading {PROXY_URL}...")
urllib.request.urlretrieve(PROXY_URL, "proxy_server.py")
print(f"Downloading {PERSONA_URL}...")
urllib.request.urlretrieve(PERSONA_URL, PERSONA_FILE)
print(f"Proxy: {os.path.getsize('proxy_server.py'):,} bytes")
print(f"Persona: {os.path.getsize(PERSONA_FILE):,} bytes")

In [ ]:
# STEP 4: Start the proxy in the background
import subprocess
import time

# Set env var for persona
env = os.environ.copy()
env["RENZ_PERSONA_PROMPT"] = open(PERSONA_FILE).read()

print("Starting WORM proxy...")
proxy_proc = subprocess.Popen(
    [sys.executable, "proxy_server.py", "--port", str(PROXY_PORT), "--no-banner"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print(f"Proxy started (PID {proxy_proc.pid})")

# Quick test
import urllib.request, json
try:
    with urllib.request.urlopen(f"http://localhost:{PROXY_PORT}/v1/models", timeout=5) as r:
        print(f"Proxy responding: {r.status}")
except Exception as e:
    print(f"Proxy test failed: {e}")

In [ ]:
# STEP 5: Start ngrok tunnel
from pyngrok import ngrok

if NGROK_AUTHTOKEN == "PASTE_YOUR_NGROK_TOKEN_HERE":
    print("ERROR: Set your ngrok authtoken in STEP 1 first")
else:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(PROXY_PORT, "http")
    public_url = tunnel.public_url
    print(f"")
    print(f"============================================")
    print(f"  YOUR PROXY IS LIVE!")
    print(f"============================================")
    print(f"")
    print(f"  Public URL:  {public_url}")
    print(f"  OpenAI API:  {public_url}/v1/chat/completions")
    print(f"  Models list: {public_url}/v1/models")
    print(f"")
    print(f"  Use this URL as your OpenAI base URL in any client.")
    print(f"  Example:")
    print(f'    export OPENAI_BASE_URL="{public_url}/v1"')
    print(f"    curl {public_url}/v1/models")
    print(f"")
    print(f"  Keep this tab open to keep the proxy alive.")
    print(f"  Stop the cell to shut down the proxy.")
    print(f"")
    # Keep the cell alive
    import time
    while True:
        time.sleep(60)
        if proxy_proc.poll() is not None:
            print("Proxy died!")
            break